# שיעור 34 — SVM: קו הפרדה, שוליים ומימוש בספרייה
2 שעות לימוד מוצעות (90 דקות),במסגרת תכנית 90 השעות המעודכנת.

**הקשר ללמידת מכונה:** פרספטרון יכול למצוא קו מפריד,אך ייתכנו כמה קווים. דודו רוצה להבין את הרעיון של העדפת שוליים ושל איזון בין הפרות למורכבות.

**עוגן:** [סיכום מפגש 11](../../13-Perceptron-SVM/Meeting-11-Didactic-Summary.md) ו[סיכום מפגש 12](../../13-Perceptron-SVM/Meeting-12-Didactic-Summary.md). יעד sex נשמר מעוגן מפגש 12. GridSearchCV נדחה; C ו־kernel קבועים מראש.

המחברת היא התאמה דידקטית חדשה. אין קובץ מקור חדש מהמרצה ולכן אין Exact-Original מלאכותי. אין לייחס לה אישור כיסוי מחוון עדכני.

## מושגים וידע קודם
Support vectors הן דוגמאות המשפיעות על הגבול. כשיש חפיפה עשויות להיות הפרות; אין טענה שכל הנתונים נפרדים. kernel מאפשר לייצג קשר לא ליניארי; מתחילים בליניארי ולא משווים גרסאות לפי Test. עבור ארבעה מאפיינים הגבול הוא על־מישור ולא קו בשרטוט דו־ממדי. אלגוריתם האימון של SVM שונה מעדכון פרספטרון.

## הסבר מתמטי
SVM ליניארי משתמש ב־w·x+b. במודל בר־הפרדה בתצורה הקנונית רוחב השוליים הוא 2/||w||. בפרשנות soft-margin,אובייקטיב ½||w||²+CΣξᵢ משלב קו ושגיאות/הפרות. C גדול מעניש הפרות יותר ביחס לרכיב המשקולות,לא מבטיח ציון Test טוב יותר. decision_function הוא ציון חתום,לא הסתברות. הרחבה זו מתארת רעיון; לא נדרשת אופטימיזציה ידנית.

## הכנה ויעד
sex כאן מתאר תיוג ביולוגי של פינגווינים במאגר,לא ניתוח אנשים. לא מסיקים מכך מודל אוניברסלי. Penguins כבר מוכר; זו הדגמה ולא מבחן בלתי תלוי חדש.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report,confusion_matrix
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
model_df=df.dropna(subset=["sex"]).copy()
features=["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
X=model_df[features].copy();y=model_df["sex"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)


## אימון והערכה
הכנה נלמדת מאימון בלבד. בניסוי השוואת C או kernel משתמשים ב־CV עם כל ה־Pipeline בתוכו,כמו בשיעור 28.

In [ ]:
# kernel ו-C נקבעו מראש; אין חיפוש על Test
model=make_pipeline(
 SimpleImputer(strategy="mean"),StandardScaler(),
 SVC(kernel="linear",C=1.0)
)
model.fit(X_train,y_train)
predictions=model.predict(X_test)
labels=["female","male"]
matrix=confusion_matrix(y_test,predictions,labels=labels)
display(pd.DataFrame(matrix,index=labels,columns=labels))
print(classification_report(y_test,predictions,labels=labels,zero_division=0))
print("Classes:",model.named_steps["svc"].classes_)
print("Number of support vectors:",model.named_steps["svc"].n_support_)


## ציון מול הסתברות
בסיווג בינרי סימן decision_function מתייחס ל־classes_[1] מול classes_[0]. הציון מושפע מהמודל ואינו אחוז ביטחון.

In [ ]:
decision_scores=model.decision_function(X_test)
print("First signed decision scores:",decision_scores[:5])
print("Positive score class:",model.named_steps["svc"].classes_[1])


## מסקנה ובדיקת הבנה
תעדו מה נכנס,מה חושב,מה התקבל ומה אי אפשר להסיק. כתבו שני משפטים על הקשר לפרויקט.
לא מעתיקים יעד species כשהעוגן הוא sex; לא מפרשים kernel כקידוד קטגוריות; לא מבטיחים שוליים ללא הפרות בכל נתון. למתקשים מקבלים Pipeline נתון ומסבירים שלבים,למצטיינים מציעים תכנית CV כתובה בלבד.

**כרטיס יציאה:** הסבירו פעולה אחת ללא קריאת הקוד ומגבלה אחת של הדוגמה. דוגמאות מומצאות אינן ראיה על אוכלוסייה אמיתית.